# 03. MLP Modeling

이 노트북은 PyTorch MLP로 `blueWins`를 예측합니다. MLP는 입력 스케일에 민감하므로 `StandardScaler`를 적용한 뒤 학습합니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.train_mlp import train_mlp

DATASETS = [
    (PROJECT_ROOT / "data" / "Challenger_Ranked_Games_10minute.csv", "10minute"),
    (PROJECT_ROOT / "data" / "Challenger_Ranked_Games_15minute.csv", "15minute"),
]

## 기본 MLP 구조

권장 기본 구조:

```text
Input → Linear(64) → ReLU → Dropout
      → Linear(32) → ReLU → Dropout
      → Linear(16) → ReLU → Dropout
      → Linear(1)
```

출력층에는 sigmoid를 직접 넣지 않고, 학습 시 `BCEWithLogitsLoss`를 사용합니다.

In [ ]:
results = []
for data_file, label in DATASETS:
    metrics = train_mlp(
        data_file=data_file,
        time_label=label,
        output_dir=PROJECT_ROOT / "results",
        model_dir=PROJECT_ROOT / "models",
        hidden_dims=(64, 32, 16),
        dropout=0.2,
        epochs=100,
        quick=True,   # 최종 실험에서는 False 권장
    )
    results.append(metrics)

pd.DataFrame(results)

## 저장된 결과 확인

생성되는 주요 파일:
- `results/tables/mlp_10minute_metrics.csv`
- `results/tables/mlp_15minute_metrics.csv`
- `results/tables/mlp_10minute_history.csv`
- `results/figures/mlp_10minute_loss_curve.png`
- `results/figures/mlp_10minute_confusion_matrix.png`
- `results/figures/mlp_10minute_roc_curve.png`

In [ ]:
comparison = pd.concat([
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "mlp_10minute_metrics.csv"),
    pd.read_csv(PROJECT_ROOT / "results" / "tables" / "mlp_15minute_metrics.csv"),
], ignore_index=True)
comparison[["model", "time_label", "accuracy", "precision", "recall", "f1", "roc_auc", "epochs_run"]]

In [ ]:
history10 = pd.read_csv(PROJECT_ROOT / "results" / "tables" / "mlp_10minute_history.csv")
history10.tail()

## 발표용 해석 방향

MLP는 피처 사이의 비선형 관계를 학습할 수 있지만, 이 데이터는 이미지나 텍스트가 아니라 표 형태의 tabular data입니다. 따라서 MLP가 항상 XGBoost보다 좋지는 않을 수 있습니다. 이 비교 자체가 프로젝트의 중요한 결론이 될 수 있습니다.